In [1]:
import os
# os.environ['http_proxy'] = "http://localhost:7890"
# os.environ['https_proxy'] = "http://localhost:7890"
# os.environ['all_proxy'] = "socks5://localhost:7890"
os.environ['HF_ENDPOINT'] = "https://hf-mirror.com"

## (1) Load model

In [2]:
import jax
import jax.numpy as jnp
from transformers import AutoTokenizer
from model import Mamba, ModelArgs # 如果你保存为 model_flax.py，修改此处

# pretrained_model_name = 'state-spaces/mamba-130m'
tokenizer = AutoTokenizer.from_pretrained('EleutherAI/gpt-neox-20b')

# 伪造配置或从本地 config.json 读取
args = ModelArgs(
    d_model=768,       # 对应 130m 的配置
    n_layer=24,
    vocab_size=50277   # gpt-neox 词表大小
)

model = Mamba(args)

# 初始化参数
key = jax.random.PRNGKey(0)
dummy_input = jnp.ones((1, 10), dtype=jnp.int32)
variables = model.init(key, dummy_input)
params = variables['params']

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


## (2) Generate Text

In [3]:
from functools import partial

@partial(jax.jit, static_argnames=('top_k',))
def forward_step(params, input_ids, top_k=40):
    logits = model.apply({'params': params}, input_ids)
    next_token_logits = logits[:, -1, :]
    return next_token_logits

def generate(params,
             tokenizer,
             prompt: str,
             n_tokens_to_gen: int = 50,
             sample: bool = True,
             top_k: int = 40,
             seed: int = 42):

    key = jax.random.PRNGKey(seed)
    input_ids = tokenizer(prompt, return_tensors='np').input_ids
    input_ids = jnp.array(input_ids)

    for _ in range(n_tokens_to_gen):
        # 获得 logits (经过 jit 的函数)
        next_token_logits = forward_step(params, input_ids, top_k)

        # Top K / Softmax 操作
        if top_k is not None:
            # 找到 top_k 阈值
            values, _ = jax.lax.top_k(next_token_logits, k=top_k)
            kth_values = values[:, -1:]
            next_token_logits = jnp.where(next_token_logits < kth_values, -1e9, next_token_logits)

        probs = jax.nn.softmax(next_token_logits, axis=-1)

        if sample:
            key, subkey = jax.random.split(key)
            next_indices = jax.random.categorical(subkey, jnp.log(probs + 1e-9), axis=-1)[:, None]
        else:
            next_indices = jnp.argmax(probs, axis=-1)[:, None]

        input_ids = jnp.concatenate([input_ids, next_indices], axis=1)

    output_completions = tokenizer.decode(input_ids[0].tolist())
    return output_completions

In [18]:
# 把 model 改成 params
print(generate(params, tokenizer, 'Mamba is the'))

Mamba is the highest mountain in the province, and also the highest and only highest mountain in the nation, and it makes it easier for me to breathe, I think. The only way to get to the top is to go with the trail.
Now let’


In [13]:
from huggingface_hub import hf_hub_download
from safetensors import safe_open
import jax.numpy as jnp

def load_flax_weights_from_hf(model_name, n_layer, flax_params):
    # 下载不依赖 torch 的 safetensors 权重文件
    model_path = hf_hub_download(repo_id=model_name, filename="model.safetensors")

    new_params = {}

    with safe_open(model_path, framework="np", device="cpu") as f:
        # 1. 提取 Embedding 和 最后的 LayerNorm
        new_params['embedding'] = {'embedding': jnp.array(f.get_tensor('backbone.embeddings.weight'))}
        new_params['norm_f'] = {'weight': jnp.array(f.get_tensor('backbone.norm_f.weight'))}

        # 2. 遍历每一层进行权重映射
        for i in range(n_layer):
            layer_name = f'layers_{i}'
            pt_prefix = f'backbone.layers.{i}'

            # PyTorch Conv1D 的 shape 是 (out_channels, in_channels/groups, kernel_size)
            # Flax Conv1D 的 shape 是 (kernel_size, in_features // groups, out_features)
            # 所以需要转置 (2, 1, 0)
            conv_weight = f.get_tensor(f'{pt_prefix}.mixer.conv1d.weight')
            conv_weight = jnp.transpose(conv_weight, (2, 1, 0))

            layer_params = {
                'norm': {
                    'weight': jnp.array(f.get_tensor(f'{pt_prefix}.norm.weight'))
                },
                'mixer': {
                    'A_log': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.A_log')),
                    'D': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.D')),
                    'in_proj': {
                        'kernel': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.in_proj.weight')).T
                    },
                    'conv1d': {
                        'kernel': conv_weight,
                        'bias': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.conv1d.bias'))
                    },
                    'x_proj': {
                        'kernel': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.x_proj.weight')).T
                    },
                    'dt_proj': {
                        'kernel': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.dt_proj.weight')).T,
                        'bias': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.dt_proj.bias'))
                    },
                    'out_proj': {
                        'kernel': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.out_proj.weight')).T
                    }
                }
            }
            new_params[layer_name] = layer_params

    return new_params

# 执行转换
pretrained_model_name = 'state-spaces/mamba-130m-hf'
params = load_flax_weights_from_hf(pretrained_model_name, args.n_layer, params)
print("预训练权重加载完成！")

预训练权重加载完成！


In [14]:
print(generate(params, tokenizer, 'John: Hi!\nSally:'))

John: Hi!
Sally: Hi!
Dana: Hi
Sally: Thanks, I'm on vacation!
Dana: Hi!
Sally: I'm at work!
Dana: So, how's it going?
Is it working well?


In [16]:
print(generate(params, tokenizer, 'The meaning of life is '))

The meaning of life is ʻinērʻ (ʻinīrʹ).  ʻērʹ (ʻēērʹ)  in the Greek  means "in", and "ēr" and "er"


In [17]:
print(generate(params, tokenizer, 'def reverse_string('))

def reverse_string(self, prefix):
    """
    Helper method for `reverse_lru_strings()`.
    """
    if prefix is not None:
        pass
    else:
        return "".join((
            self.__
